In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/awatanshsingh/eta-zoned/train_features.parquet
/kaggle/input/datasets/awatanshsingh/eta-zoned/dev_features.parquet
/kaggle/input/datasets/awatanshsingh/eta-zoned/sample_features.parquet


In [2]:
# import pandas as pd

# TRAIN_PATH = "/kaggle/input/datasets/awatanshsingh/eta-zoned/train_features.parquet"
# DEV_PATH   = "/kaggle/input/datasets/awatanshsingh/eta-zoned/dev_features.parquet"

# # Use columns selectively if memory issue
# traindf = pd.read_parquet(TRAIN_PATH)
# devdf   = pd.read_parquet(DEV_PATH)

# print(traindf.shape, devdf.shape)

In [3]:
# drop_cols = [
#     'duration_seconds',
#     'requested_at',
#     'pickup_zone_name', 'dropoff_zone_name',
#     'pickup_centroid_source', 'dropoff_centroid_source'
# ]

# target = 'duration_seconds'
# features = [col for col in traindf.columns if col not in drop_cols]

# X_train = traindf[features].copy()
# y_train = traindf[target]

# X_dev = devdf[features].copy()
# y_dev = devdf[target]

# cat_cols = ['pickup_borough', 'dropoff_borough']

# for col in cat_cols:
#     X_train[col] = X_train[col].astype('category')
#     X_dev[col] = X_dev[col].astype('category')
    
#     # align categories (CRITICAL)
#     X_dev[col] = X_dev[col].cat.set_categories(X_train[col].cat.categories)

In [4]:
# import numpy as np

# def add_features(df):
#     df['lat_diff'] = df['pickup_latitude'] - df['dropoff_latitude']
#     df['lon_diff'] = df['pickup_longitude'] - df['dropoff_longitude']
    
#     # cyclic time encoding
#     df['hour_sin'] = np.sin(2 * np.pi * df['request_hour'] / 24)
#     df['hour_cos'] = np.cos(2 * np.pi * df['request_hour'] / 24)
    
#     return df

# X_train = add_features(X_train)
# X_dev = add_features(X_dev)

In [5]:
# import lightgbm as lgb

# model = lgb.LGBMRegressor(
#     device='gpu',              # ⭐ GPU
#     gpu_platform_id=0,
#     gpu_device_id=0,
    
#     n_estimators=2000,
#     learning_rate=0.03,
#     num_leaves=128,
    
#     subsample=0.8,
#     colsample_bytree=0.8,
    
#     random_state=42,
#     n_jobs=-1
# )

In [6]:
# model.fit(
#     X_train,
#     y_train,
#     eval_set=[(X_dev, y_dev)],
#     eval_metric='mae',
#     categorical_feature=cat_cols,
#     callbacks=[
#         lgb.early_stopping(100),
#         lgb.log_evaluation(50)
#     ]
# )

In [10]:
# ================================
# 🚀 0. SETUP
# ================================
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

TRAIN_PATH = "/kaggle/input/datasets/awatanshsingh/eta-zoned/train_features.parquet"
DEV_PATH   = "/kaggle/input/datasets/awatanshsingh/eta-zoned/dev_features.parquet"

TARGET = "duration_seconds"

DROP_COLS = [
    'duration_seconds',
    'requested_at',
    'pickup_zone_name', 'dropoff_zone_name',
    'pickup_centroid_source', 'dropoff_centroid_source'
]

CAT_COLS = ['pickup_borough', 'dropoff_borough']


# ================================
# ⚡ FEATURE ENGINEERING (DISABLED)
# ================================
def add_features(df):
    return df


# ================================
# ⚡ 1. LOAD TRAIN
# ================================
print("Loading TRAIN...")
traindf = pd.read_parquet(TRAIN_PATH)
traindf = add_features(traindf)


# ================================
# ⚡ 2. PREP TRAIN
# ================================
features = [c for c in traindf.columns if c not in DROP_COLS]

X_train = traindf[features].copy()
y_train = traindf[TARGET].copy()

# ✅ FORCE categorical correctly (robust)
for col in CAT_COLS:
    X_train[col] = pd.Categorical(X_train[col])

# memory optimization
float_cols = X_train.select_dtypes(include=['float64']).columns
int_cols   = X_train.select_dtypes(include=['int64']).columns

X_train[float_cols] = X_train[float_cols].astype('float32')
X_train[int_cols]   = X_train[int_cols].astype('int32')


# ================================
# ⚡ 3. LOAD DEV
# ================================
print("Loading DEV...")
devdf = pd.read_parquet(DEV_PATH)
devdf = add_features(devdf)

# ✅ enforce SAME columns/order safely
X_dev = devdf.reindex(columns=features).copy()
y_dev = devdf[TARGET].copy()

# ✅ categorical alignment (FIXED PROPERLY)
for col in CAT_COLS:
    X_dev[col] = pd.Categorical(
        X_dev[col],
        categories=X_train[col].cat.categories
    )

# memory optimization
float_cols_dev = X_dev.select_dtypes(include=['float64']).columns
int_cols_dev   = X_dev.select_dtypes(include=['int64']).columns

X_dev[float_cols_dev] = X_dev[float_cols_dev].astype('float32')
X_dev[int_cols_dev]   = X_dev[int_cols_dev].astype('int32')


# ================================
# 🚀 4. LIGHTGBM DATASET
# ================================
train_data = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=CAT_COLS,
    free_raw_data=True
)

valid_data = lgb.Dataset(
    X_dev,
    label=y_dev,
    reference=train_data
)


# ================================
# 🚀 5. GPU OPTIMIZED PARAMS
# ================================
params = {
    "objective": "regression",
    "metric": "mae",
    
    # GPU
    "device": "gpu",
    "gpu_use_dp": False,
    
    # speed
    "max_bin": 255,
    
    # workload ↑ (better GPU usage)
    "learning_rate": 0.05,
    "num_leaves": 256,
    "max_depth": -1,
    
    # regularization
    "min_data_in_leaf": 100,
    
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    
    "verbosity": -1,
    "seed": 42
}


# ================================
# 🚀 6. TRAIN
# ================================
print("Training...")

model = lgb.train(
    params,
    train_data,
    num_boost_round=3000,
    valid_sets=[valid_data],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100)
    ]
)


# ================================
# 📊 7. EVALUATE
# ================================
print("Evaluating...")

y_pred = model.predict(X_dev)
mae = mean_absolute_error(y_dev, y_pred)

print(f"\n🔥 FINAL MAE: {mae:.4f}")

Loading TRAIN...
Loading DEV...
Training...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l1: 288.535
Early stopping, best iteration is:
[73]	valid_0's l1: 287.2
Evaluating...

🔥 FINAL MAE: 287.1995


In [11]:
import pickle

# --- Save (after training) ---
# Native LightGBM format (recommended)
model.save_model("lgbm_model.txt")

# Optional: pickle
with open("lgbm_model.pkl", "wb") as f:
    pickle.dump(model, f)